# MIMII fan exploration

Scaffold for exploring the MIMII fan recordings in `data/6_dB/fan/` 

**Dataset notes**: MIMII clips are 10 s @ 16 kHz. Directory layout is `6_dB/fan/id_XX/{normal,abnormal}/NNNNNNNN.wav`, `6_dB` is the SNR of the factory-noise mix, `id_XX` is an individual machine.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import wavfile
from scipy import signal


def find_fan_dir() -> Path:
    base = Path.cwd()
    candidate = base / ".." / "data" / "6_dB" / "fan"
    if candidate.is_dir():
        return candidate
    raise FileNotFoundError(
        f"data/6_dB/fan not found in or above cwd={Path.cwd()} — "
    )


FAN_DIR = find_fan_dir()
print(f"data: {FAN_DIR}")

# Pipeline defaults (keep in sync with backend .env.example)
SAMPLE_RATE = 16_000
WINDOW_SECONDS = 1.0

plt.rcParams["figure.figsize"] = (12, 3)

Inventory

In [ ]:
clips = pd.DataFrame(
    {
        "path": p,
        "machine": p.parts[-3],   # id_00 ...
        "label": p.parts[-2],     # normal / abnormal
        "clip": p.stem,
    }
    for p in sorted(FAN_DIR.glob("id_*/*/*.wav"))
)
# An empty glob yields a DataFrame with no columns and confusing KeyErrors downstream
assert not clips.empty, f"no wav files matched under {FAN_DIR}"
print(f"{len(clips)} clips")
clips.groupby(["machine", "label"]).size().unstack(fill_value=0)

In [ ]:
def load_clip(path: Path) -> tuple[int, np.ndarray]:
    """
    drop all but ch 0
    """
    rate, samples = wavfile.read(path)
    if samples.ndim > 1:
        samples = samples[:, 0]
    if np.issubdtype(samples.dtype, np.integer):
        samples = samples / np.iinfo(samples.dtype).max
    return rate, samples.astype(np.float32)

sample = clips.groupby(["machine", "label"]).head(2)
checks = []
for _, row in sample.iterrows():
    rate, raw = wavfile.read(row.path)
    checks.append({
        "machine": row.machine, "label": row.label, "clip": row["clip"],
        "rate": rate, "seconds": raw.shape[0] / rate,
        "channels": 1 if raw.ndim == 1 else raw.shape[1], "dtype": str(raw.dtype),
    })
pd.DataFrame(checks)

Waveforms

is amplitude alone interesting (do abnormal clips just get louder)

In [ ]:
MACHINE = "id_00"  # TODO: repeat for the other ids

fig, axes = plt.subplots(2, 2, figsize=(14, 5), sharex=True, sharey=True)
for row, label in enumerate(["normal", "abnormal"]):
    picks = clips.query("machine == @MACHINE and label == @label").head(2)
    for ax, (_, clip) in zip(axes[row], picks.iterrows()):
        rate, samples = load_clip(clip.path)
        t = np.arange(len(samples)) / rate
        ax.plot(t, samples, linewidth=0.3)
        ax.set_title(f"{clip.machine} {label} {clip['clip']}", fontsize=9)
fig.supxlabel("seconds"); fig.supylabel("amplitude")
fig.tight_layout()

Spectrograms

In [ ]:
NFFT = 1024  # TODO: try 512/2048 to trade time vs frequency resolution

def spectrogram_db(samples: np.ndarray, rate: int):
    f, t, sxx = signal.spectrogram(samples, fs=rate, nperseg=NFFT, noverlap=NFFT // 2)
    return f, t, 10 * np.log10(sxx + 1e-12)

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for ax, label in zip(axes, ["normal", "abnormal"]):
    clip = clips.query("machine == @MACHINE and label == @label").iloc[0]
    rate, samples = load_clip(clip.path)
    f, t, db = spectrogram_db(samples, rate)
    im = ax.pcolormesh(t, f, db, shading="gouraud", cmap="magma")
    ax.set_title(f"{clip.machine} {label} {clip['clip']}")
    ax.set_xlabel("seconds")
axes[0].set_ylabel("Hz")
fig.colorbar(im, ax=axes, label="dB")

Band energy

In [ ]:
# TODO: band edges made up, try variations
BAND_EDGES_HZ = [0, 250, 500, 1000, 2000, 4000, 8000]
BAND_LABELS = [f"{lo}-{hi}" for lo, hi in zip(BAND_EDGES_HZ, BAND_EDGES_HZ[1:])]

def band_energies(samples: np.ndarray, rate: int) -> np.ndarray:
    """Mean energy per band over the whole clip (dB)."""
    f, _, sxx = signal.spectrogram(samples, fs=rate, nperseg=NFFT, noverlap=NFFT // 2)
    idx = np.digitize(f, BAND_EDGES_HZ[1:-1])
    return np.array([10 * np.log10(sxx[idx == b].mean() + 1e-12) for b in range(len(BAND_LABELS))])

rows = []
for _, clip in clips.query("machine == @MACHINE").groupby("label").head(20).iterrows():
    rate, samples = load_clip(clip.path)
    rows.append({"label": clip.label, **dict(zip(BAND_LABELS, band_energies(samples, rate)))})
bands = pd.DataFrame(rows)

fig, axes = plt.subplots(1, len(BAND_LABELS), figsize=(14, 3), sharey=True)
for ax, band in zip(axes, BAND_LABELS):
    bands.boxplot(column=band, by="label", ax=ax)
    ax.set_title(band); ax.set_xlabel("")
fig.suptitle(f"band energy (dB) by label {MACHINE}"); fig.tight_layout()

Windows

are features consistant across a normal clip's windows? E.g. are there 1s windows in a 10s "abnormal" clip which look normal.

In [ ]:
def windows(samples: np.ndarray, rate: int, seconds: float = WINDOW_SECONDS):
    n = int(rate * seconds)
    for i in range(len(samples) // n):
        yield i, samples[i * n : (i + 1) * n]

clip = clips.query("machine == @MACHINE and label == 'normal'").iloc[0]
rate, samples = load_clip(clip.path)
per_window = pd.DataFrame(
    {"window": i, "rms": float(np.sqrt(np.mean(w**2))), **dict(zip(BAND_LABELS, band_energies(w, rate)))}
    for i, w in windows(samples, rate)
)
per_window.set_index("window")[BAND_LABELS].plot(marker="o", title=f"per-window band energy {clip.machine} normal {clip['clip']}")
plt.ylabel("dB");

Distance test

fit "normal" from normal windows only, then score everything else by how far it sits from that.
Half the normal clips fit the baseline, the other half are held out; every abnormal clip is held out.

In [ ]:
normal = clips.query("machine == @MACHINE and label == 'normal'").sample(frac=1, random_state=0)
fit_clips, test_clips = normal.iloc[: len(normal) // 2], normal.iloc[len(normal) // 2 :]
abnormal = clips.query("machine == @MACHINE and label == 'abnormal'")

def window_features(paths):
    rows = []
    for path in paths:
        rate, samples = load_clip(path)
        for _, w in windows(samples, rate):
            rows.append(band_energies(w, rate))
    return np.array(rows)

X_fit, X_test, X_abn = window_features(fit_clips.path), window_features(test_clips.path), window_features(abnormal.path)
X_fit.shape, X_test.shape, X_abn.shape

In [ ]:
# "normal" = mean and covariance of the fit windows; score = Mahalanobis distance from it
mu = X_fit.mean(axis=0)
cov_inv = np.linalg.pinv(np.cov(X_fit, rowvar=False))

def score(X):
    d = X - mu
    return np.sqrt(np.sum((d @ cov_inv) * d, axis=1))

s_fit, s_test, s_abn = score(X_fit), score(X_test), score(X_abn)

plt.hist(s_test, bins=60, alpha=0.6, label="held-out normal windows")
plt.hist(s_abn, bins=60, alpha=0.6, label="abnormal windows")
plt.xlabel("distance from normal (std devs)"); plt.ylabel("windows"); plt.title(MACHINE); plt.legend();

Does it rank abnormal above normal? AUROC = P(random abnormal scores higher than random normal); 0.5 is chance.

In [ ]:
from sklearn.metrics import roc_auc_score

def auroc(neg, pos):
    return roc_auc_score(np.r_[np.zeros(len(neg)), np.ones(len(pos))], np.r_[neg, pos])

# clip score = mean of its windows (every clip is 10 s, so equal window counts)
c_test = s_test.reshape(len(test_clips), -1).mean(axis=1)
c_abn = s_abn.reshape(len(abnormal), -1).mean(axis=1)
print(f"window AUROC {auroc(s_test, s_abn):.3f}   clip AUROC {auroc(c_test, c_abn):.3f}")

Threshold: alarm above the p-th percentile of the fit windows' own scores (they are all normal, so this limits false alarms)

In [ ]:
for p in [90, 95, 99, 99.5]:
    thr = np.percentile(s_fit, p)
    print(f"p{p:<5} threshold {thr:5.2f}   false alarms {np.mean(s_test > thr):.3f}   detected {np.mean(s_abn > thr):.3f}")

Does id_00's "normal" describe the other fans? Score their normal windows against the same baseline (a shared baseline would false-alarm if these sit high).

In [ ]:
rows = [{"machine": MACHINE, "median distance": np.median(s_test)}]
for m in sorted(clips.machine.unique()):
    if m == MACHINE:
        continue
    paths = clips.query("machine == @m and label == 'normal'").path.head(50)  # TODO: all clips
    rows.append({"machine": m, "median distance": np.median(score(window_features(paths)))})
pd.DataFrame(rows).set_index("machine").round(2)

TODO
- inspect the worst false negatives by ear/spectrogram
- repeat with the other ids as the fitted machine
- overlapping windows; log-spaced bands instead of the made-up edges above
- other machine types (pump, valve)